# **Preprocessing, EDA, dan BERTopic**

Alur: (1) satu blok **preprocessing terpadu** dari `raw.csv` -> checkpoint `sentences.csv`; (2) **EDA** atas hasilnya; (3) analisis **stopword**; (4) **Fase 03: BERTopic** memetakan kalimat ke topik (dasar penamaan aspek FiQA). Teks yang di-embed tetap natural (tanpa stemming/stopword/lowercase).

## **1. Import & baca data mentah**

In [1]:
import re
import unicodedata
from pathlib import Path
from collections import Counter
import pandas as pd

raw = pd.read_csv("data/raw/raw.csv", dtype={"id_artikel": str}, keep_default_na=False)
required = {"id_artikel", "judul", "isi_mentah", "timestamp"}
if not required.issubset(raw.columns) or raw["id_artikel"].duplicated().any() or raw["id_artikel"].eq("").any():
    raise ValueError("Kolom wajib tidak lengkap, atau id_artikel kosong/duplikat.")
print(f"Artikel mentah: {len(raw):,} | isi kosong: {raw['isi_mentah'].str.strip().eq('').sum():,}")

Artikel mentah: 3,979 | isi kosong: 0


## **2. Preprocessing terpadu**

Satu blok: deteksi bank per artikel, pecah kalimat (lindungi singkatan/inisial/desimal), bersihkan noise (dateline, URL, footer, disclaimer, promosi tautan, **listing saham/kurs**), beri **flag review**, lalu pisahkan **teks unik** (untuk BERTopic) vs **semua kemunculan** (untuk dipetakan balik ke artikel). Setiap kalimat yang dibuang dicatat `alasan_dibuang` untuk audit.

In [2]:
# --- pola & kamus ---
POLA_BANK = {
    "BCA":     r"\bbca\b|\bbbca\b|bank central asia",
    "BRI":     r"\bbri\b|\bbbri\b|bank rakyat indonesia|\bbrilink\b",
    "BNI":     r"\bbni\b|\bbbni\b|bank negara indonesia",
    "Mandiri": r"\bmandiri\b|\bbmri\b|bank mandiri",
}
# singkatan yang titiknya JANGAN dianggap akhir kalimat (+ gelar yang muncul di data)
SINGKATAN = ["PT", "Tbk", "Persero", "Rp", "US", "No", "Jl", "Dr", "Ir", "Hj",
             "Prof", "Sdr", "S.E", "M.M", "Ph.D", "dll", "dsb", "dkk", "a.l", "yakni"]

SPLIT      = re.compile(r'(?<=[.!?])\s+|(?<=[.!?]["\u201d])\s+')
DATELINE   = re.compile(r'^\s*(?:jakarta\s*,?\s*)?cnbc\s+indonesia\s*[-\u2013\u2014:]\s*', re.I)
URL        = re.compile(r'(?i)\b(?:https?://|www\.)[^\s<>"\u201d]+')
INVISIBLE  = re.compile('[\u200b\u200c\u200d\u200e\u200f\u2060\ufeff]')
FOOTER     = re.compile(r'^(?:cnbc\s+indonesia\s+research(?:\s+market@cnbcindonesia\.com)?|selamat\s+berinvestasi\s+secara\s+bijak)[.!\s]*$', re.I)
LINK_PROMO = re.compile(r'(?i)(?:baca\s+artikel\s+cnbc\s+indonesia|selengkapnya\s+di\s+sini).*(?:cnbcindonesia\.com|download\s+apps\s+cnbc)')
DISCLAIMER = re.compile(r"^keputusan berinvestasi sepenuhnya berada di tangan masing-masing investor sesuai dengan profil risiko dan tujuan keuangan pribadi[.!\s]*$", re.I)
FRAGMENT   = re.compile(r"^(?:ltd\.?|\(?dok\.?)$", re.I)
NUMERIC    = re.compile(r'^[\W\d_]+$', re.UNICODE)
SAMPAH     = re.compile(r"CNBC\s*INDONESIA\s*RESEARCH|market@cnbcindonesia\.com|\bNEXT\b|Lihat\s*juga|Saksikan\s+(?:Video|juga)|\bBaca\s*:|^Foto\s*:|^Infografis\s*:", re.I)

# listing saham / tabel kurs: kalimat yang padat kode-emiten+nominal, atau label kurs
KODE_EMITEN = re.compile(r"\([A-Z]{4}\)")
RP_NOMINAL  = re.compile(r"Rp[\s.]?\d")
KURS_LABEL  = re.compile(r"(?i)\b(?:e-?rate|tt\s*counter|bank\s*notes|special\s*rates?|ttb|tts)\b|-?\s*(?:beli|jual)\s*:")

def is_listing_noise(text):
    if len(KURS_LABEL.findall(text)) >= 3:                               # tabel kurs
        return True
    if len(KODE_EMITEN.findall(text)) >= 3 and len(RP_NOMINAL.findall(text)) >= 3:  # daftar net buy/sell
        return True
    if "Rekomendasi Saham" in text or len(re.findall(r"\|", text)) >= 2 or re.search(r"\bTP\b.*\bSL\b", text):
        return True
    return False

def deteksi_bank(teks):
    return ", ".join(b for b, pat in POLA_BANK.items() if re.search(pat, teks, flags=re.I))

def pecah_kalimat(teks):
    t = re.sub(r"\s+", " ", str(teks)).strip()
    for ab in SINGKATAN:
        t = re.sub(rf"\b{re.escape(ab)}\.", lambda m: m.group(0)[:-1] + "<DOT>", t)
    t = re.sub(r"\b([A-Z])\.(?=\s)", r"\1<DOT>", t)      # inisial nama
    t = re.sub(r"(?<=\d)\.(?=\d)", "<DOT>", t)           # desimal/ribuan
    return [x.replace("<DOT>", ".").strip() for x in SPLIT.split(t) if x.strip()]

def bersihkan(text):
    text = unicodedata.normalize("NFC", str(text))
    text = INVISIBLE.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    if LINK_PROMO.search(text):           return "", "promosi_tautan"
    text = DATELINE.sub("", text)
    text = re.sub(r'^\s*[\u2013\u2014\u2022\u00b7-]+\s*', '', text)   # dash/bullet pembuka
    text = URL.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text:                          return "", "kosong_setelah_bersih"
    if FOOTER.fullmatch(text):            return "", "footer"
    if DISCLAIMER.fullmatch(text):        return "", "disclaimer_investasi"
    if FRAGMENT.fullmatch(text):          return "", "fragmen_nonkalimat"
    if NUMERIC.fullmatch(text):           return "", "nomor_atau_tanda_saja"
    if SAMPAH.search(text):               return "", "boilerplate"
    if is_listing_noise(text):            return "", "listing_saham_atau_kurs"
    return text, ""

# --- jalankan pecah + bersih ---
rows = []
for row in raw.itertuples(index=False):
    banks = deteksi_bank(f"{row.judul} {row.isi_mentah}")
    for i, asli in enumerate(pecah_kalimat(row.isi_mentah)):
        bersih, alasan = bersihkan(asli)
        rows.append({"id_artikel": row.id_artikel, "id_kalimat": f"{row.id_artikel}_{i}",
                     "bank": banks, "timestamp": row.timestamp,
                     "kalimat_asli": asli, "kalimat": bersih, "alasan_dibuang": alasan})
semua = pd.DataFrame(rows)
dibuang = semua[semua.alasan_dibuang.ne("")].copy()
occ = semua[semua.alasan_dibuang.eq("")].copy()
occ["n_kata"] = occ.kalimat.str.split().str.len()

# buang kalimat terlalu pendek (<4) & terlalu panjang (>80) dengan alasan tercatat
occ.loc[occ.n_kata.lt(4), "alasan_dibuang"] = "terlalu_pendek"
occ.loc[occ.n_kata.gt(80), "alasan_dibuang"] = "terlalu_panjang"
dibuang = pd.concat([dibuang, occ[occ.alasan_dibuang.ne("")]], ignore_index=True)
occ = occ[occ.alasan_dibuang.eq("")].copy()

# flag review (BUKAN auto-delete) untuk sisa yang perlu dilihat manual
occ["flag_konteks_daftar"] = occ.kalimat.str.contains(r"(?i)\b(?:tbk|persero)\b.*(?:rp|us\$).*\b\d{1,2}[.)]\s*$", regex=True, na=False)
occ["flag_sisa_markup"]    = occ.kalimat.str.contains(r"(?i)\[gambas:|\b(?:foto|infografis)\s*:|\bbaca\s+(?:juga|artikel)\b", regex=True, na=False)

# teks unik (input BERTopic) vs semua kemunculan (map balik ke artikel)
occ["id_teks"] = occ.groupby("kalimat", sort=False)["id_kalimat"].transform("first")
unique = occ.drop_duplicates("id_teks").copy()
counts = occ.groupby("id_teks").size().rename("jumlah_kemunculan")
unique = unique.merge(counts, left_on="id_teks", right_index=True, validate="one_to_one")
assert occ.id_kalimat.is_unique and unique.id_teks.is_unique
assert len(occ) == int(unique.jumlah_kemunculan.sum())

print(f"Potongan awal : {len(semua):,}")
print(f"Dibuang       : {len(dibuang):,}")
print(dibuang.alasan_dibuang.value_counts().to_string())
print(f"Kemunculan    : {len(occ):,} | teks unik: {len(unique):,}")

Potongan awal : 82,509
Dibuang       : 2,450
alasan_dibuang
listing_saham_atau_kurs    1192
nomor_atau_tanda_saja       425
terlalu_pendek              346
footer                      170
terlalu_panjang             156
boilerplate                  77
disclaimer_investasi         47
fragmen_nonkalimat           30
promosi_tautan                7
Kemunculan    : 80,059 | teks unik: 63,634


## **3. Checkpoint & file audit**

In [3]:
OUT = Path("data/processed/bertopic_prep")
OUT.mkdir(parents=True, exist_ok=True)

# checkpoint utama (teks unik, kolom ramping) -> kompatibel dengan tahap lanjut
unique[["id_artikel", "id_kalimat", "bank", "timestamp", "kalimat", "n_kata"]].to_csv(
    "data/processed/sentences.csv", index=False)

# file lengkap + audit
occ.to_csv(OUT / "sentences_bertopic_occurrences.csv", index=False, encoding="utf-8-sig")
unique.to_csv(OUT / "sentences_bertopic_unique.csv", index=False, encoding="utf-8-sig")
dibuang.to_csv(OUT / "dibuang_preprocessing.csv", index=False, encoding="utf-8-sig")
occ.loc[occ.kalimat.ne(occ.kalimat_asli), ["id_artikel", "id_kalimat", "kalimat_asli", "kalimat"]].to_csv(
    OUT / "perubahan_preprocessing.csv", index=False, encoding="utf-8-sig")
occ.loc[occ[["flag_konteks_daftar", "flag_sisa_markup"]].any(axis=1)].to_csv(
    OUT / "perlu_review.csv", index=False, encoding="utf-8-sig")
print(f"[CHECKPOINT] sentences.csv: {len(unique):,} kalimat unik")
print("File audit ->", OUT.resolve())

[CHECKPOINT] sentences.csv: 63,634 kalimat unik
File audit -> C:\erick\Kulyeah\Semester 5\Text mining\Bank-News-Topic-Modelling-Aspect-Based-Sentimental-Analysis\data\processed\bertopic_prep


## **4. EDA**

In [4]:
# dipakai untuk analisis di bawah
kal = unique

print("Statistik jumlah kata per kalimat:")
print(kal["n_kata"].describe().round(1))

Statistik jumlah kata per kalimat:
count    63634.0
mean        18.5
std          8.7
min          4.0
25%         12.0
50%         17.0
75%         23.0
max         80.0
Name: n_kata, dtype: float64


In [5]:
print("Distribusi panjang kalimat (kata):")
bins = [0, 5, 10, 15, 20, 30, 50, 10000]
print(pd.cut(kal["n_kata"], bins=bins, right=False).value_counts().sort_index())

Distribusi panjang kalimat (kata):
n_kata
[0, 5)           232
[5, 10)         6444
[10, 15)       16843
[15, 20)       16152
[20, 30)       17490
[30, 50)        6008
[50, 10000)      465
Name: count, dtype: int64


### Duplikat (sebelum dedup unik) & kemunculan terbanyak

In [6]:
print("Teks identik dengan kemunculan terbanyak (belum tentu noise):")
for row in kal.sort_values("jumlah_kemunculan", ascending=False).head(10).itertuples():
    print(f"  {row.jumlah_kemunculan:>3}x | {row.kalimat[:90]}")

Teks identik dengan kemunculan terbanyak (belum tentu noise):
   27x | Nasabah BRI Prioritas bisa mendapatkan BRI Prioritas Debit Card yang menyediakan semua lay
   23x | Mengutip Stockbit, berikut net foreign sell perdagangan Rabu!
   18x | Yakni welcome bonus berupa cashback senilai Rp 1,5 juta setelah aktivasi dan transaksi per
   18x | "Perusahaan para cukong dan keluarga Soeharto merupakan sasaran utama pembakaran dan penja
   18x | Nasabah istimewa BNI Emerald akan mendapatkan produk dan layanan yang dirancang khusus ses
   18x | Keuntungan bagi nasabah prioritas Bank Mandiri dapat dirasakan sejak bergabung.
   18x | "Dalam sekejap, seluruh mobil di garasi terbakar, termasuk juga seisi rumah.
   18x | Mengutip Stockbit, berikut net foreign sell perdagangan Senin!
   18x | Kemudian, manfaat spesial bagi nasabah Mandiri Prioritas adalah 4x Livin'poin untuk transa
   18x | Selain itu, ada perlindungan dari pembelian barang, perlindungan asuransi perjalanan bebas


### Rentang waktu & cakupan bank

In [7]:
print("Rentang tanggal:", raw.timestamp.min(), "-", raw.timestamp.max())
print("\nJumlah bank yang disebut per kalimat:")
print(kal["bank"].map(lambda x: len(x.split(", ")) if x else 0).value_counts().sort_index())
print("Kalimat tanpa bank terdeteksi:", (kal["bank"] == "").sum())

Rentang tanggal: 2024-10-21 - 2026-10-20

Jumlah bank yang disebut per kalimat:
bank
0       44
1    30448
2    12581
3    13742
4     6819
Name: count, dtype: int64
Kalimat tanpa bank terdeteksi: 44


### Contoh kalimat pendek (cek bukan sampah)

In [8]:
print("Contoh kalimat terpendek yang dipertahankan (>=4 kata):")
for s in kal.nsmallest(20, "n_kata")["kalimat"].tolist():
    print("  |", s)

Contoh kalimat terpendek yang dipertahankan (>=4 kata):
  | BCA menjadi yang terparah.
  | (Kompas, 15 Mei 1998).
  | Hati-hati akun palsu ya!"
  | Berikut kisah perjalanan keduanya.
  | Warisan Jumbo Michael Hartono!
  | "Itu kan sovereign rating.
  | Sisanya baru consumer goods.
  | Kedua, kelompok masyarakat bawah.
  | "NIM menjadi semakin menantang.
  | Belakangan, kepercayaan itu terbukti.
  | "Keputusan BI sudah tepat.
  | Kedua, ialah soal fiskal.
  | Enggak bisa seperti itu.
  | Termasuk mall Plaza Senayan.
  | "Saham cenderung ngikutin harga.
  | "Masih terkait isu eksternal.
  | Pada perdagangan Rabu (1/10/2025).
  | Perbedaan ini terlihat jelas.
  | USD Yield hingga 4,13%.
  | "Kami terus menjaga rupiah.


## **5. Stopword untuk keyword topik**

Stopword dipakai HANYA untuk representasi keyword c-TF-IDF, bukan teks yang di-embed. Final = (generik Sastrawi + domain finansial) DIKURANGI kata negasi (negasi menentukan arah sentimen).

In [9]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# frekuensi token (lowercase hanya untuk analisis)
tokens = []
for k in kal["kalimat"]:
    tokens += re.findall(r"[a-zA-Z]+", str(k).lower())
freq = Counter(tokens)
print(f"Total token: {len(tokens):,} | kata unik: {len(freq):,}")
print("20 kata tersering:")
for w, c in freq.most_common(20):
    print(f"  {w:12} {c}")

Total token: 1,121,116 | kata unik: 20,688
20 kata tersering:
  dan          23233
  yang         22927
  di           17876
  saham        17496
  rp           16723
  dengan       13092
  pada         12906
  ini          12277
  bank         11916
  dari         10690
  indonesia    8924
  dalam        8826
  pt           8483
  miliar       8467
  menjadi      8121
  tbk          7186
  triliun      7147
  untuk        6791
  ihsg         6763
  pasar        6648


In [10]:
stop_generik = set(StopWordRemoverFactory().get_stop_words())
stop_domain = {"rp", "pt", "tbk", "persero", "miliar", "triliun", "juta", "ribu",
               "i", "ii", "iii", "yoy", "as", "us",
               "senin", "selasa", "rabu", "kamis", "jumat", "sabtu", "minggu",
               "indonesia", "menjadi", "ihsg"}
NEGASI = {"tidak", "tak", "bukan", "belum", "jangan", "tanpa", "kurang",
          "non", "anti", "minus", "gagal", "batal", "turun", "melemah", "koreksi"}
stopwords_id = sorted((stop_generik | stop_domain) - NEGASI)
assert not (set(stopwords_id) & NEGASI)
print(f"Stopword generik: {len(stop_generik)} | domain: {len(stop_domain)} | negasi dipertahankan: {len(NEGASI)}")
print(f"Stopword final untuk vectorizer: {len(stopwords_id)} (negasi sudah dikecualikan)")

Stopword generik: 123 | domain: 24 | negasi dipertahankan: 15
Stopword final untuk vectorizer: 144 (negasi sudah dikecualikan)


---
## **Fase 03 - BERTopic (Aspect Extraction)**

Input = kalimat **unik** dari preprocessing di atas. `min_cluster_size=300` agar cluster lebih sedikit & padu (lebih mudah dipetakan ke aspek FiQA). `random_state=42` -> reproducible.

In [11]:
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

SEED = 42
MIN_CLUSTER_SIZE = 300
MIN_SAMPLES = 10
MIN_DF = 5
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

sent = unique.copy().reset_index(drop=True)
docs = sent["kalimat"].tolist()
if not docs or any(not isinstance(d, str) or not d.strip() for d in docs):
    raise ValueError("Input BERTopic kosong atau mengandung kalimat kosong.")
print(f"Input BERTopic: {len(docs):,} kalimat unik")

Input BERTopic: 63,634 kalimat unik


In [12]:
# 1. Embedding
embedding_model = SentenceTransformer(MODEL_NAME)
embeddings = embedding_model.encode(docs, show_progress_bar=True, batch_size=64)

# 2. UMAP (seed tetap)
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric="cosine", random_state=SEED)

# 3. HDBSCAN
hdbscan_model = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES,
                        metric="euclidean", cluster_selection_method="eom", prediction_data=True)

# 4. Representasi topik
vectorizer_model = CountVectorizer(stop_words=stopwords_id, ngram_range=(1, 2), min_df=MIN_DF)

# 5. BERTopic
topic_model = BERTopic(embedding_model=embedding_model, umap_model=umap_model,
                       hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model,
                       language="multilingual", calculate_probabilities=False, verbose=True)
topics, probs = topic_model.fit_transform(docs, embeddings)
if len(topics) != len(sent):
    raise ValueError("Jumlah topik != jumlah kalimat input.")

info = topic_model.get_topic_info()
n_out = int((pd.Series(topics) == -1).sum())
print(f"Topik non-outlier: {sum(info.Topic != -1)} | outlier: {n_out:,} ({n_out/len(topics):.1%})")
info.head(30)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/995 [00:00<?, ?it/s]

2026-11-17 20:26:41,970 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-17 20:27:51,039 - BERTopic - Dimensionality - Completed ✓
2026-11-17 20:27:51,041 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-17 20:27:58,201 - BERTopic - Cluster - Completed ✓
2026-11-17 20:27:58,219 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-17 20:28:00,515 - BERTopic - Representation - Completed ✓


Topik non-outlier: 36 | outlier: 20,506 (32.2%)


,Topic,Count,Name,Representation,Representative_Docs
0,-1,20506,-1_bank_tersebut_dana_tahun,"[bank, tersebut, dana, tahun, kredit, lebih, b...",[Adapun saat ini pemerintah akan menarik Rp 20...
1,0,5058,0_saham gabungan_gabungan_indeks harga_harga s...,"[saham gabungan, gabungan, indeks harga, harga...",[Indeks Harga Saham Gabungan (IHSG) masih bang...
2,1,3441,1_pasar_saham_asing_investor,"[pasar, saham, asing, investor, the, fed, the ...",[Investor asing tercatat melakukan aksi jual b...
3,2,2821,2_bbca_indeks poin_poin_bbri,"[bbca, indeks poin, poin, bbri, indeks, saham,...",[Saham Bank Rakyat Indonesia (BBRI) menyumbang...
4,3,1989,3_bri_umkm_hery_terus,"[bri, umkm, hery, terus, memperkuat, layanan, ...","[""BRI sangat powerful ya, apalagi begitu melih..."
5,4,1890,4_pasar_asing_investor asing_seluruh pasar,"[pasar, asing, investor asing, seluruh pasar, ...","[Sementara itu, investor asing tercatat masih ..."
6,5,1714,5_pertumbuhan_ekonomi_bank_bank mandiri,"[pertumbuhan, ekonomi, bank, bank mandiri, man...",[Langkah ini sejalan dengan peran Bank Mandiri...
7,6,1708,6_msci_pasar_market_ekonomi,"[msci, pasar, market, ekonomi, global, saham, ...",[Tekanan ini tidak lepas dari pengumuman MSCI ...
8,7,1675,7_sektor_konsumer_primer_baku,"[sektor, konsumer, primer, baku, zona, utilita...",[Nyaris seluruh sektor perdagangan bergerak di...
9,8,1479,8_2026_2025_dividen_tanggal,"[2026, 2025, dividen, tanggal, rapat, dijadwal...",[Cum date dividen untuk pasar reguler dan nego...


### Preview keyword + kalimat representatif

In [15]:
for t in info["Topic"].tolist():
    if t == -1:
        continue
    keyword = ", ".join(word for word, _ in (topic_model.get_topic(t) or [])[:8])
    print(f"\n=== Topik {t} (n={int(info.loc[info.Topic == t, 'Count'].iloc[0])}) ===")
    print("Kata kunci:", keyword)
    for text in (topic_model.get_representative_docs(t) or [])[:3]:
        print(" -", text)


=== Topik 0 (n=5058) ===
Kata kunci: saham gabungan, gabungan, indeks harga, harga saham, harga, perdagangan, level, indeks
 - Indeks Harga Saham Gabungan (IHSG) masih bangkit dari zona merah.
 - Indeks Harga Saham Gabungan (IHSG) ditutup di zona merah pada perdagangan kemarin.
 - Indeks Harga Saham Gabungan (IHSG) kembali melemah.

=== Topik 1 (n=3441) ===
Kata kunci: pasar, saham, asing, investor, the, fed, the fed, inflasi
 - Investor asing tercatat melakukan aksi jual bersih (net foreign sell) di sejumlah saham pada perdagangan Senin (9/3/2026), di tengah tekanan yang terjadi di pasar saham domestik.
 - Data tersebut memperkuat keyakinan bahwa bank sentral AS (The Federal Reserve/The Fed) akan mempertahankan suku bunga acuannya pada pertemuan akhir Januari ini, meskipun terdapat tekanan politik dari Gedung Putih untuk segera menurunkan suku bunga.
 - Selain itu, dari global, keputusan tentang suku bunga The Federal Reserve (The Fed) akan menjadi sorotan meskipun baru akan diumumka

### Simpan hasil topik (unik & per kemunculan/artikel)

In [16]:
sent["topic"] = topics
result = occ.merge(sent[["id_teks", "topic"]], on="id_teks", how="left", validate="many_to_one")
if result.topic.isna().any() or len(result) != len(occ):
    raise ValueError("Ada kemunculan tanpa topik atau jumlah baris berubah.")
sent.to_csv(OUT / "sentences_unique_with_topic.csv", index=False, encoding="utf-8-sig")
result.to_csv(OUT / "sentences_occurrences_with_topic.csv", index=False, encoding="utf-8-sig")
info.to_csv(OUT / "topic_info_baru.csv", index=False, encoding="utf-8-sig")
print(f"Kemunculan terpetakan: {len(result):,} | artikel: {result.id_artikel.nunique():,}")

Kemunculan terpetakan: 80,059 | artikel: 3,979


### **Langkah manual berikutnya**

Namai & gabungkan topik menjadi **10-15 aspek**, rujuk **FiQA Corporate & Stock**; topik tak jelas -> "Lainnya". Dokumentasikan di `daftar_aspek.md`, lalu **kunci daftar aspek** sebelum anotasi (Fase 04). Pemetaan topik->aspek dikerjakan manual oleh anotator.